In [1]:
import scarf

scarf.configure_output(level='WARNING', progress=True)

dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    'tenx_5K_pbmc_rnaseq',
    destination='scarf_datasets',
    zarr=True,
)
ds = scarf.DataStore(
    f'{dataset}/data.zarr',
    nthreads=4,
    min_features_per_cell=10,
)
ds.filter_cells(
    attrs=['RNA_nCounts', 'RNA_nFeatures'],
    highs=[15000, 4000],
    lows=[1000, 500],
    reset_previous=True,
)
if 'I__hvgs' not in ds.RNA.feats.columns:
    ds.mark_hvgs(min_cells=20, top_n=500, show_plot=False)

Downloading bucket files: 38209567 / 38209567 complete

Downloading bytes: 38209567 / 38209567 complete

In [2]:
normalized = ds.run_normalization(
    feat_key='hvgs',
    update_state=False,
)
pca = ds.run_pca(normalized, dims=15, update_state=False)
ann = ds.build_ann_index(pca, update_state=False)
neighbors_k11 = ds.query_neighbors(ann, k=11, update_state=False)
graph_k11 = ds.build_connectivity_map(neighbors_k11, update_state=False)

Writing data: 1 / 1 complete

Fitting PCA: 1 / 1 complete

Writing reduced coordinates: 1 / 1 complete

Calculating reduced coordinates: 1 / 1 complete

Fitting ANN: 1 / 1 complete

Identifying neighbors: 1 / 1 complete

In [3]:
neighbors_k15 = ds.query_neighbors(ann, k=15, update_state=False)
graph_k15 = ds.build_connectivity_map(neighbors_k15, update_state=False)

print('normalization reused:', ds.run_normalization(feat_key='hvgs', update_state=False) == normalized)
print('PCA reused:', ds.run_pca(normalized, dims=15, update_state=False) == pca)
print('ANN index reused:', ds.build_ann_index(pca, update_state=False) == ann)
print('neighbors recomputed:', neighbors_k15 != neighbors_k11)
print('graph recomputed:', graph_k15 != graph_k11)

Identifying neighbors: 1 / 1 complete

normalization reused: True
PCA reused: True
ANN index reused: True
neighbors recomputed: True
graph recomputed: True


In [4]:
pca_dims20 = ds.run_pca(normalized, dims=20, update_state=False)
ann_dims20 = ds.build_ann_index(pca_dims20, update_state=False)
neighbors_dims20 = ds.query_neighbors(ann_dims20, k=11, update_state=False)
graph_dims20 = ds.build_connectivity_map(neighbors_dims20, update_state=False)

print('PCA recomputed:', pca_dims20 != pca)
print('ANN index recomputed:', ann_dims20 != ann)
print('neighbors recomputed:', neighbors_dims20 != neighbors_k11)
print('graph recomputed:', graph_dims20 != graph_k11)
print('normalization reused:', ds.run_normalization(feat_key='hvgs', update_state=False) == normalized)

Fitting PCA: 1 / 1 complete

Writing reduced coordinates: 1 / 1 complete

Calculating reduced coordinates: 1 / 1 complete

Fitting ANN: 1 / 1 complete

Identifying neighbors: 1 / 1 complete

PCA recomputed: True
ANN index recomputed: True
neighbors recomputed: True
graph recomputed: True
normalization reused: True


In [5]:
forced = ds.run_normalization(
    feat_key='hvgs',
    update_state=False,
    invalidate_cache=True,
)
status = ds.inspect_artifact(forced)
print('new artifact:', forced != normalized)
print('complete:', status.complete)
print('operation:', status.operation)

Writing data: 1 / 1 complete

new artifact: True
complete: True
operation: run_normalization


In [6]:
lineage = ds.lineage(
    {
        'k11 graph': graph_k11,
        'k15 graph': graph_k15,
    }
)
print(lineage)

ArtifactLineage(outputs=2, artifacts=11, dependencies=14)


In [7]:
print(lineage.to_mermaid())

flowchart LR
    artifact0["datastore / cell_selection | filter_cells | 2d0b9e246fe7"]
    artifact1["RNA / feature_selection | mark_hvgs | a0a1e6adf628"]
    artifact2["datastore / cell_selection | filter_cells | 86f558c8a747"]
    artifact3["RNA / normalized | run_normalization | ddea479f6361"]
    artifact4["RNA / feature_scaling | calculate_feature_scaling | 7552c9266a14"]
    artifact5["RNA / reduction | run_pca | e7c39438019f"]
    artifact6["RNA / ann_index | build_ann_index | 5678222b08da"]
    artifact7["RNA / neighbors | query_neighbors | 76e06eb8e834"]
    artifact8["RNA / connectivity_map | build_connectivity_map | afaf45e9be51 | outputs: k11 graph"]
    artifact9["RNA / neighbors | query_neighbors | faf86fea6ba1"]
    artifact10["RNA / connectivity_map | build_connectivity_map | 1f0e921809cd | outputs: k15 graph"]
    artifact0 -->|"cell_selection"| artifact1
    artifact9 -->|"neighbors"| artifact10
    artifact1 -->|"feature_selection"| artifact3
    artifact2 -->|"cell_